# SONIC-Lite G1: simple Colab smoke test

This notebook mounts Google Drive, pulls the project from its GitHub remote, locates BONES-SEED, and runs one forward pass, one backward pass, and one motor-token inference check.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YigitGunduc/robot.git'
REPO_DIR = Path('/content/robot')

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Using repository:', REPO_DIR)
print(subprocess.check_output(['git', 'remote', '-v'], text=True).splitlines()[0])

In [ ]:
DATASET_ROOT = Path('/content/drive/MyDrive/Datasets/bones-seed')
CSV_ROOT = DATASET_ROOT / 'g1' / 'csv'

print('Dataset root:', DATASET_ROOT)
print('Dataset root exists:', DATASET_ROOT.exists())
print('Extracted CSV root exists:', CSV_ROOT.exists())

if DATASET_ROOT.exists():
    for path in sorted(DATASET_ROOT.iterdir()):
        print(path.name)

if CSV_ROOT.exists():
    csv_files = list(CSV_ROOT.rglob('*.csv'))
    print('CSV files found:', len(csv_files))
else:
    print('The archive must be extracted before running dataset conversion.')

In [ ]:
# RSL-RL supplies the MLP actor base class; Colab already includes PyTorch.
%pip install -q rsl-rl-lib tensordict

In [ ]:
import sys
import torch
from tensordict import TensorDict

sys.path.insert(0, str(REPO_DIR))
from sonic_lite_g1.model import SonicLiteActor

torch.manual_seed(0)
batch_size = 1
future_dim = 5 * 64
proprio_dim = 640  # smoke-test shape; the live mjlab config supplies the real history size

obs = TensorDict({
    'future': torch.randn(batch_size, future_dim),
    'proprio': torch.randn(batch_size, proprio_dim),
}, batch_size=[batch_size])

actor = SonicLiteActor(
    obs=obs,
    obs_groups={'actor': ['future', 'proprio']},
    obs_set='actor',
    output_dim=29,
    hidden_dims=(32, 16),
    distribution_cfg={
        'class_name': 'GaussianDistribution',
        'init_std': 0.5,
        'std_type': 'scalar',
    },
)
print('Actor created')

In [ ]:
# One forward pass and one backward pass.
action = actor(obs)
loss = action.mean()
loss.backward()

finite_gradients = all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in actor.parameters()
)
print('Forward action shape:', tuple(action.shape))
print('Backward loss:', float(loss))
print('Finite gradients:', finite_gradients)
assert action.shape == (1, 29)
assert finite_gradients

In [ ]:
# One inference/token check.
with torch.no_grad():
    token = actor.encode_motor_token(obs)
    inference_action = actor(obs)

print('Motor token shape:', tuple(token.shape))
print('Motor token range:', float(token.min()), float(token.max()))
print('Inference action shape:', tuple(inference_action.shape))
assert token.shape == (1, 64)
assert torch.isfinite(token).all()
print('SMOKE TEST PASSED')